# 03 – XGBoost Training, Evaluation & Deployment Preparation
End-to-end training notebook for the final production model.

## Objectives
- Train the final XGBoost model
- Tune hyperparameters
- Evaluate performance
- Analyze errors
- Export a deployment-ready pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib
import json 

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor

import matplotlib.pyplot as plt


In [2]:
df_model = pd.read_csv("../data/processed.csv", dtype={"fuel_type": str})

df_model.to_parquet("data.parquet", engine = "pyarrow")

df_model = pd.read_parquet("data.parquet", engine = "pyarrow", )

df_model.dtypes

price             float64
miles             float64
year              float64
make                  str
body_type             str
vehicle_type          str
drivetrain            str
transmission          str
fuel_type             str
engine_size       float64
engine_block          str
city                  str
province              str
miles_was_zero      int64
car_age           float64
miles_per_year    float64
model_trim            str
region                str
dtype: object

Pasting code from N02 -> Preprocessing Pipelines, defining X and y, target and predictor variables as well Column transformer

In [3]:
TARGET = "price"

X = df_model.drop(columns=[TARGET]).copy()
y = df_model[TARGET].copy()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42)

In [5]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()


In [6]:
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

X[categorical_features].nunique().sort_values(ascending=False)

model_trim      5998
region           963
city             758
make              57
fuel_type         24
body_type         21
province          15
engine_block       3
drivetrain         3
vehicle_type       2
transmission       2
dtype: int64

In [7]:
high_cardinality_features = [
    "model_trim",
    "region",
    "city"
]

In [8]:
cat_columns_to_drop = [
    'model_trim' , 'region' , 'city' ,
    'model', 'trim'
]

low_cardinality_features = [
    col for col in categorical_cols
    if col not in cat_columns_to_drop
]

# High-cardinality categorical columns with separate transformers
model_trim_features = ["model_trim"]
region_features = ["region"]
city_features = ["city"]

# Final selected features
selected_features = (
    numeric_cols
    + low_cardinality_features
    + model_trim_features
    + region_features
    + city_features
)

selected_features = list(dict.fromkeys(selected_features))

# Remove accidental duplicates
selected_features = list(dict.fromkeys(selected_features))

In [9]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]
)

low_cardinality_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

#
model_trim_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ))
])

region_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

city_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

preprocessor = ColumnTransformer(transformers=[
    ("numerical", numeric_transformer, numeric_cols),

    ("low_cardinality_categorical",
     low_cardinality_categorical_transformer,
     low_cardinality_features),

    ("model_trim",
     model_trim_transformer,
     model_trim_features),

    ("region",
     region_transformer,
     region_features),

    ("city",
     city_transformer,
     city_features)
])

After copy-pasting the code it is now time to define the model

In [ ]:
xgb_model = XGBRegressor(

    
)
model = {"XGBregressor": xgb_model}